In [68]:
import pandas as pd
import random

In [69]:
class SimpleRLProductionSystem:
    def __init__(self, max_inventory, max_production, max_demand, epsilon=0.1):
        self.max_inventory = max_inventory
        self.max_production = max_production
        self.max_demand = max_demand
        self.epsilon = epsilon
        self.data = pd.DataFrame(columns=['State', 'Demand', 'Action', 'Reward'])
        self._initialize_data()

    def _initialize_data(self):
        rows = []
        for state in range(self.max_inventory + 1):
            for demand in range(self.max_demand + 1):
                for action in range(self.max_production + 1):
                    rows.append({'State': state, 'Demand': demand, 'Action': action, 'Reward': float('-inf')})
        self.data = pd.concat([self.data, pd.DataFrame(rows)], ignore_index=True)

    def choose_action(self, state, demand):
        if random.uniform(0, 1) < self.epsilon:
            return random.randint(0, self.max_production)  # Explore
        else:
            state_demand_data = self.data[(self.data['State'] == state) & (self.data['Demand'] == demand)]
            return state_demand_data.loc[state_demand_data['Reward'].idxmax()]['Action']  # Exploit

    def get_reward(self, state, action, demand):
        potential_inventory = state + action
        if potential_inventory > self.max_inventory:
            return -20  # Penalize for exceeding the capacity
        else:
            if potential_inventory >= demand:
                return 10 - (potential_inventory - demand)  # Positive reward for meeting demand, less for overproduction
            else:
                return -10 + (potential_inventory - demand)  # Negative reward for underproduction

    def update_data(self, state, demand, action, reward):
        row_index = self.data[(self.data['State'] == state) & (self.data['Demand'] == demand) & (self.data['Action'] == action)].index
        if self.data.at[row_index[0], 'Reward'] < reward:
            self.data.at[row_index[0], 'Reward'] = reward

    def simulate(self, episodes):
        for _ in range(episodes):
            state = 0  # Start each episode with the state set to 0
            demand = random.randint(0, self.max_demand)  # Randomly chosen demand
            action = self.choose_action(state, demand)
            reward = self.get_reward(state, action, demand)
            self.update_data(state, demand, action, reward)

    def evaluate_agent(self, episodes):
        total_reward = 0
        demand_met_count = 0

        for episode in range(episodes):
            state = 0
            demand = random.randint(0, self.max_demand)
            action = self.choose_action_test(state, demand)
            reward = self.get_reward(state, action, demand)
            total_reward += reward

            if state + action >= demand:
                demand_met_count += 1

            # Printing the demand and action for each episode
            print(f"Episode {episode + 1}: Demand = {demand}, Action = {action}")

        average_reward = total_reward / episodes
        demand_met_rate = demand_met_count / episodes

        return average_reward, demand_met_rate

    
    def choose_action_test(self, state, demand):
            state_demand_data = self.data[(self.data['State'] == state) & (self.data['Demand'] == demand)]
            return state_demand_data.loc[state_demand_data['Reward'].idxmax()]['Action']  # Exploit

In [67]:
# Example usage
system = SimpleRLProductionSystem(max_inventory=10, max_production=10, max_demand=8)
system.simulate(episodes=100000)
print(system.data)
system.data.to_csv('file1.csv')

     State Demand Action  Reward
0        0      0      0    10.0
1        0      0      1     9.0
2        0      0      2     8.0
3        0      0      3     7.0
4        0      0      4     6.0
...    ...    ...    ...     ...
1084    10      8      6    -inf
1085    10      8      7    -inf
1086    10      8      8    -inf
1087    10      8      9    -inf
1088    10      8     10    -inf

[1089 rows x 4 columns]


In [64]:
# Evaluate the trained agent and observe output directly from the method
evaluation_episodes = 10  # Set to a smaller number for demonstration
average_reward, demand_met_rate = system.evaluate_agent(evaluation_episodes)

# Print summary statistics
print(f"\nAverage Reward over {evaluation_episodes} evaluation episodes: {average_reward}")
print(f"Demand Met Rate over {evaluation_episodes} evaluation episodes: {demand_met_rate:.2%}")

Episode 1: Demand = 0, Action = 0
Episode 2: Demand = 4, Action = 4
Episode 3: Demand = 7, Action = 7
Episode 4: Demand = 1, Action = 1
Episode 5: Demand = 2, Action = 2
Episode 6: Demand = 8, Action = 8
Episode 7: Demand = 2, Action = 2
Episode 8: Demand = 0, Action = 0
Episode 9: Demand = 7, Action = 7
Episode 10: Demand = 4, Action = 4

Average Reward over 10 evaluation episodes: 10.0
Demand Met Rate over 10 evaluation episodes: 100.00%
